# 🎯 PNG Preprocessing Pipeline - After DICOM Windowing

**Objectif:** Compter les PNG, mapper aux labels, splitter train/val/test, et préparer le PyTorch DataLoader

**Input:** 200 PNG files (3-channel RGB) + `dataset_labeled_enriched.csv`

**Output:** PyTorch DataLoaders prêts pour l'entraînement

## 📦 Section 1: Installation des dépendances et configuration Colab

In [ ]:
# Install required libraries
import subprocess
import sys

packages = ['torch', 'torchvision', 'pandas', 'numpy', 'pillow', 'scikit-learn']
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ All packages installed successfully!")

In [ ]:
# Import libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import json
from datetime import datetime

print("✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## 🔌 Section 2: Montage de Google Drive et configuration des chemins

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define paths (adjust according to your Drive structure)
DRIVE_PATH = '/content/drive/MyDrive'  # Change if needed
PNG_INPUT_FOLDER = os.path.join(DRIVE_PATH, 'preprocessed_images_multiwindow')  # Output from windowing
DATASET_CSV = os.path.join(DRIVE_PATH, 'dataset_labeled_enriched.csv')  # Labels mapping
OUTPUT_FOLDER = os.path.join(DRIVE_PATH, 'png_preprocessing_output')  # Results folder

# Create output folder
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"📁 PNG Input Folder: {PNG_INPUT_FOLDER}")
print(f"📁 Dataset CSV: {DATASET_CSV}")
print(f"📁 Output Folder: {OUTPUT_FOLDER}")
print(f"\n✅ Google Drive mounted and folders configured!")

## 🔍 Section 3: Comptage et inventaire des PNG

In [ ]:
# 📊 Count and analyze PNG files
print(f"\n{'='*60}")
print("📊 PNG FILE INVENTORY")
print(f"{'='*60}")

# List all PNG files
png_files = [f for f in os.listdir(PNG_INPUT_FOLDER) if f.lower().endswith('.png')]
png_files.sort()

print(f"\n✅ Total PNG files found: {len(png_files)}")

# Calculate statistics
file_sizes = []
for file in png_files:
    filepath = os.path.join(PNG_INPUT_FOLDER, file)
    size_bytes = os.path.getsize(filepath)
    size_mb = size_bytes / (1024 * 1024)
    file_sizes.append((file, size_mb))

file_sizes.sort(key=lambda x: x[1], reverse=True)

# Statistics
sizes_mb = [size for _, size in file_sizes]
total_size_mb = sum(sizes_mb)
avg_size_mb = np.mean(sizes_mb)
min_size_mb = np.min(sizes_mb)
max_size_mb = np.max(sizes_mb)

print(f"\n📈 SIZE STATISTICS:")
print(f"   Total size: {total_size_mb:.2f} MB")
print(f"   Average size: {avg_size_mb:.2f} MB")
print(f"   Min size: {min_size_mb:.2f} MB")
print(f"   Max size: {max_size_mb:.2f} MB")

# Create inventory DataFrame
inventory_data = []
for filename, size_mb in file_sizes:
    inventory_data.append({
        'filename': filename,
        'size_mb': round(size_mb, 3),
        'path': os.path.join(PNG_INPUT_FOLDER, filename)
    })

df_inventory = pd.DataFrame(inventory_data)

print(f"\n📋 First 10 PNG files:")
print(df_inventory[['filename', 'size_mb']].head(10).to_string(index=False))

# Save inventory CSV
inventory_csv = os.path.join(OUTPUT_FOLDER, 'png_inventory.csv')
df_inventory.to_csv(inventory_csv, index=False)
print(f"\n✅ PNG inventory saved to: {inventory_csv}")

## 🏷️ Section 4: Chargement du dataset et création du mapping PNG → Labels

In [ ]:
# Load dataset with labels
print(f"\n{'='*60}")
print("🏷️  LABEL MAPPING")
print(f"{'='*60}")

df_labels = pd.read_csv(DATASET_CSV)
print(f"\n📋 Dataset shape: {df_labels.shape}")
print(f"   Columns: {list(df_labels.columns)}")
print(f"\n   First 5 rows:")
print(df_labels.head().to_string())

In [ ]:
# Create PNG to label mapping
# Assumption: PNG filenames contain the original DICOM ID
# e.g., "IM-0001-0001_multiwindow.png" → find row with "IM-0001-0001" in dataset

print(f"\n🔗 Creating PNG → Label mapping...")

mapping_data = []
png_to_label = {}
matched_count = 0
unmatched_files = []

for png_file in png_files:
    # Extract DICOM ID from PNG filename
    # Pattern: "IM-0001-0001_multiwindow.png" → "IM-0001-0001"
    base_name = png_file.replace('_multiwindow.png', '').replace('.png', '')
    
    # Try to find matching row in dataset
    # Check if any column contains this ID
    matching_rows = df_labels[df_labels.astype(str).apply(lambda x: x.str.contains(base_name, na=False)).any(axis=1)]
    
    if len(matching_rows) > 0:
        # Get the label from the first matching row
        # Assuming there's a 'finding' or 'label' column
        label_col = None
        if 'finding' in df_labels.columns:
            label_col = 'finding'
        elif 'label' in df_labels.columns:
            label_col = 'label'
        elif 'diagnosis' in df_labels.columns:
            label_col = 'diagnosis'
        else:
            # Take the last column as label
            label_col = df_labels.columns[-1]
        
        label = matching_rows[label_col].iloc[0]
        
        mapping_data.append({
            'png_filename': png_file,
            'dicom_id': base_name,
            'label': label
        })
        png_to_label[png_file] = label
        matched_count += 1
    else:
        unmatched_files.append(png_file)

df_mapping = pd.DataFrame(mapping_data)

print(f"\n✅ Mapping results:")
print(f"   Matched: {matched_count}/{len(png_files)}")
print(f"   Unmatched: {len(unmatched_files)}")

if len(unmatched_files) > 0:
    print(f"\n⚠️  Unmatched PNG files (first 5):")
    for f in unmatched_files[:5]:
        print(f"   - {f}")

print(f"\n📋 Label distribution:")
label_counts = df_mapping['label'].value_counts()
print(label_counts)

# Save mapping CSV
mapping_csv = os.path.join(OUTPUT_FOLDER, 'png_label_mapping.csv')
df_mapping.to_csv(mapping_csv, index=False)
print(f"\n✅ PNG-Label mapping saved to: {mapping_csv}")

## ✂️ Section 5: Train/Validation/Test Split avec stratification

In [ ]:
print(f"\n{'='*60}")
print("✂️  TRAIN/VALIDATION/TEST SPLIT")
print(f"{'='*60}")

# Use only matched files for training
if len(df_mapping) == 0:
    print("❌ ERROR: No matched PNG files found!")
else:
    # First split: 80% train+val, 20% test
    df_train_val, df_test = train_test_split(
        df_mapping,
        test_size=0.20,
        random_state=42,
        stratify=df_mapping['label']
    )
    
    # Second split: 70% train, 30% validation (of the train_val set)
    # This gives us: 56% train, 24% val, 20% test
    df_train, df_val = train_test_split(
        df_train_val,
        test_size=0.3,
        random_state=42,
        stratify=df_train_val['label']
    )
    
    # Calculate percentages
    train_pct = len(df_train) / len(df_mapping) * 100
    val_pct = len(df_val) / len(df_mapping) * 100
    test_pct = len(df_test) / len(df_mapping) * 100
    
    print(f"\n📊 Split results:")
    print(f"   Train: {len(df_train)} files ({train_pct:.1f}%)")
    print(f"   Val:   {len(df_val)} files ({val_pct:.1f}%)")
    print(f"   Test:  {len(df_test)} files ({test_pct:.1f}%)")
    print(f"   Total: {len(df_mapping)} files")
    
    # Check class distribution in each split
    print(f"\n🔄 Class distribution verification:")
    for split_name, split_df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
        print(f"\n   {split_name}:")
        dist = split_df['label'].value_counts()
        for label, count in dist.items():
            pct = count / len(split_df) * 100
            print(f"      {label}: {count} ({pct:.1f}%)")
    
    # Save split CSVs
    train_csv = os.path.join(OUTPUT_FOLDER, 'split_train.csv')
    val_csv = os.path.join(OUTPUT_FOLDER, 'split_val.csv')
    test_csv = os.path.join(OUTPUT_FOLDER, 'split_test.csv')
    
    df_train.to_csv(train_csv, index=False)
    df_val.to_csv(val_csv, index=False)
    df_test.to_csv(test_csv, index=False)
    
    print(f"\n✅ Split CSVs saved:")
    print(f"   {train_csv}")
    print(f"   {val_csv}")
    print(f"   {test_csv}")

## 🔧 Section 6: PyTorch Dataset et DataLoader

In [ ]:
# Define PyTorch Dataset class
class DicomPNGDataset(Dataset):
    """Dataset pour charger les images PNG multi-windowing en PyTorch"""
    
    def __init__(self, dataframe, image_folder, transform=None):
        """
        Args:
            dataframe: DataFrame with columns ['png_filename', 'dicom_id', 'label']
            image_folder: Path to folder containing PNG images
            transform: Transformations to apply (torchvision.transforms)
        """
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_folder = image_folder
        self.transform = transform
        
        # Create label to index mapping
        self.unique_labels = sorted(self.dataframe['label'].unique())
        self.label_to_idx = {label: idx for idx, label in enumerate(self.unique_labels)}
        self.idx_to_label = {idx: label for label, idx in self.label_to_idx.items()}
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        
        # Load image
        img_path = os.path.join(self.image_folder, row['png_filename'])
        image = Image.open(img_path).convert('RGB')
        
        # Apply transforms if any
        if self.transform:
            image = self.transform(image)
        
        # Get label as integer
        label = self.label_to_idx[row['label']]
        
        return image, label, row['png_filename']

print("✅ DicomPNGDataset class defined!")

In [ ]:
# Define transforms for each split
# Training: with augmentation
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # Standard size for CNN models
    transforms.RandomRotation(10),  # Random rotation ±10°
    transforms.RandomHorizontalFlip(0.3),  # 30% horizontal flip
    transforms.RandomVerticalFlip(0.2),  # 20% vertical flip
    transforms.ColorJitter(brightness=0.1, contrast=0.1),  # Slight color jitter
    transforms.ToTensor(),  # Convert to tensor (0-1)
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalize
])

# Validation and Test: no augmentation, only preprocessing
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print("✅ Transforms defined!")
print(f"\n   Train transforms: Augmentation enabled")
print(f"   Val/Test transforms: No augmentation")

In [ ]:
# Create Dataset instances
print(f"\n{'='*60}")
print("📦 CREATING PYTORCH DATASETS")
print(f"{'='*60}")

train_dataset = DicomPNGDataset(
    df_train,
    PNG_INPUT_FOLDER,
    transform=train_transforms
)

val_dataset = DicomPNGDataset(
    df_val,
    PNG_INPUT_FOLDER,
    transform=val_test_transforms
)

test_dataset = DicomPNGDataset(
    df_test,
    PNG_INPUT_FOLDER,
    transform=val_test_transforms
)

print(f"\n✅ Datasets created:")
print(f"   Train: {len(train_dataset)} images")
print(f"   Val: {len(val_dataset)} images")
print(f"   Test: {len(test_dataset)} images")
print(f"\n📊 Unique classes: {train_dataset.unique_labels}")
print(f"   Total classes: {len(train_dataset.unique_labels)}")

In [ ]:
# Create DataLoaders
print(f"\n{'='*60}")
print("🔄 CREATING PYTORCH DATALOADERS")
print(f"{'='*60}")

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"\n✅ DataLoaders created:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

## 🧪 Section 7: Vérification et visualisation des données

In [ ]:
# Test DataLoader - Load one batch
print(f"\n{'='*60}")
print("🧪 TESTING DATALOADER")
print(f"{'='*60}")

# Get one batch from train loader
images, labels, filenames = next(iter(train_loader))

print(f"\n✅ Batch loaded successfully:")
print(f"   Images shape: {images.shape}  # (batch, channels, height, width)")
print(f"   Labels: {labels}")
print(f"   First 3 filenames: {filenames[:3]}")

# Verify image properties
print(f"\n📏 Image properties:")
print(f"   Min value: {images.min():.4f}")
print(f"   Max value: {images.max():.4f}")
print(f"   Mean: {images.mean():.4f}")
print(f"   Std: {images.std():.4f}")

In [ ]:
# Visualization of sample images
import matplotlib.pyplot as plt

# Get 4 sample images from train set
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

for i in range(4):
    idx = np.random.randint(0, len(train_dataset))
    image, label, filename = train_dataset[idx]
    
    # Convert tensor back to image for visualization
    # Reverse normalization: x_original = (x_normalized + 1) / 2
    image_np = image.numpy().transpose(1, 2, 0)
    image_np = (image_np + 1) / 2  # Denormalize
    image_np = np.clip(image_np, 0, 1)
    
    label_name = train_dataset.idx_to_label[label]
    
    axes[i].imshow(image_np)
    axes[i].set_title(f"Label: {label_name}")
    axes[i].set_xlabel(filename)
    axes[i].axis('off')

plt.tight_layout()
viz_path = os.path.join(OUTPUT_FOLDER, 'sample_images.png')
plt.savefig(viz_path, dpi=100, bbox_inches='tight')
plt.show()

print(f"✅ Sample images visualization saved to: {viz_path}")

## 💾 Section 8: Sauvegarde de la configuration et metadata

In [ ]:
# Save configuration and metadata
config = {
    'timestamp': datetime.now().isoformat(),
    'png_input_folder': PNG_INPUT_FOLDER,
    'dataset_csv': DATASET_CSV,
    'total_png_files': len(png_files),
    'matched_files': len(df_mapping),
    'unmatched_files': len(unmatched_files),
    'train_size': len(df_train),
    'val_size': len(df_val),
    'test_size': len(df_test),
    'batch_size': BATCH_SIZE,
    'num_classes': len(train_dataset.unique_labels),
    'class_names': train_dataset.unique_labels,
    'label_to_idx': train_dataset.label_to_idx,
    'image_size': 224,
    'normalization': {
        'mean': [0.5, 0.5, 0.5],
        'std': [0.5, 0.5, 0.5]
    }
}

# Save configuration as JSON
config_json = os.path.join(OUTPUT_FOLDER, 'preprocessing_config.json')
with open(config_json, 'w') as f:
    json.dump(config, f, indent=4)

print(f"✅ Configuration saved to: {config_json}")
print(f"\n📋 Configuration Summary:")
print(f"   Total PNG files: {config['total_png_files']}")
print(f"   Matched files: {config['matched_files']}")
print(f"   Train samples: {config['train_size']}")
print(f"   Val samples: {config['val_size']}")
print(f"   Test samples: {config['test_size']}")
print(f"   Classes: {config['num_classes']}")
print(f"   Class names: {config['class_names']}")

In [ ]:
# Create summary report
summary_report = f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 PNG PREPROCESSING PIPELINE - SUMMARY REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📊 FILE STATISTICS
   Total PNG files: {config['total_png_files']}
   Total size: {total_size_mb:.2f} MB
   Average size: {avg_size_mb:.2f} MB

🏷️  LABEL MAPPING
   Successfully mapped: {config['matched_files']} files
   Unmatched: {config['unmatched_files']} files
   Number of classes: {config['num_classes']}
   Classes: {', '.join(config['class_names'])}

✂️  DATA SPLIT
   Train set:      {config['train_size']} samples ({config['train_size']/config['matched_files']*100:.1f}%)
   Validation set: {config['val_size']} samples ({config['val_size']/config['matched_files']*100:.1f}%)
   Test set:       {config['test_size']} samples ({config['test_size']/config['matched_files']*100:.1f}%)
   Total:          {config['matched_files']} samples

🔧 DATALOADER CONFIGURATION
   Batch size: {config['batch_size']}
   Image size: {config['image_size']}x{config['image_size']}
   Number of channels: 3 (RGB)
   Normalization: mean {config['normalization']['mean']}, std {config['normalization']['std']}
   Train batches: {len(train_loader)}
   Val batches: {len(val_loader)}
   Test batches: {len(test_loader)}

📁 OUTPUT FILES
   PNG inventory: {os.path.basename(inventory_csv)}
   Label mapping: {os.path.basename(mapping_csv)}
   Train split: {os.path.basename(train_csv)}
   Val split: {os.path.basename(val_csv)}
   Test split: {os.path.basename(test_csv)}
   Config: {os.path.basename(config_json)}
   Sample visualization: sample_images.png

✅ PIPELINE COMPLETE!
   All datasets and dataloaders are ready for model training.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

print(summary_report)

# Save report
report_path = os.path.join(OUTPUT_FOLDER, 'PREPROCESSING_REPORT.txt')
with open(report_path, 'w') as f:
    f.write(summary_report)

print(f"✅ Report saved to: {report_path}")

## 🚀 Section 9: Code pour accéder aux DataLoaders dans l'entraînement

In [ ]:
# Example code to use DataLoaders in training loop

example_training_code = '''
# EXAMPLE: Using DataLoaders in training

import torch.nn as nn
import torch.optim as optim

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = YourModel().to(device)  # Replace with your model
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    # Training phase
    model.train()
    for batch_idx, (images, labels, filenames) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (batch_idx + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
    
    # Validation phase
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels, _ in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        val_accuracy = 100 * correct / total
        print(f'Epoch [{epoch+1}/{num_epochs}], Validation Accuracy: {val_accuracy:.2f}%')

# Test phase
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels, filenames in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
print(f'Test Accuracy: {test_accuracy:.2f}%')
'''

print("📝 EXAMPLE TRAINING CODE:")
print(example_training_code)

# Save example code
example_code_path = os.path.join(OUTPUT_FOLDER, 'example_training_loop.py')
with open(example_code_path, 'w') as f:
    f.write(example_training_code)

print(f"\n✅ Example code saved to: {example_code_path}")

In [ ]:
# Final summary with next steps
next_steps = f"""
🎉 PREPROCESSING COMPLETE! 

📌 NEXT STEPS FOR MODEL TRAINING:

1️⃣  Choose a model architecture (e.g., ResNet, EfficientNet, ViT)
2️⃣  Implement the model class
3️⃣  Define loss function (CrossEntropyLoss for multi-class)
4️⃣  Choose optimizer (Adam, SGD, etc.)
5️⃣  Run training loop using train_loader, val_loader, test_loader
6️⃣  Monitor performance with metrics (Accuracy, Precision, Recall, F1)

📊 AVAILABLE OBJECTS IN MEMORY:
   - train_loader: DataLoader for training
   - val_loader: DataLoader for validation
   - test_loader: DataLoader for testing
   - train_dataset: DicomPNGDataset for train split
   - val_dataset: DicomPNGDataset for val split
   - test_dataset: DicomPNGDataset for test split
   - Number of classes: {config['num_classes']}
   - Class names: {config['class_names']}

🔗 ALL OUTPUT FILES:
   Location: {OUTPUT_FOLDER}
   - preprocessing_config.json (configuration)
   - png_inventory.csv (all PNG files)
   - png_label_mapping.csv (image-label mapping)
   - split_train.csv (training set metadata)
   - split_val.csv (validation set metadata)
   - split_test.csv (test set metadata)
   - PREPROCESSING_REPORT.txt (this report)

💡 TIPS:
   - Use GPU for training: device = torch.device('cuda')
   - Try data augmentation in transforms for better generalization
   - Use learning rate scheduling (ReduceLROnPlateau)
   - Save best model: torch.save(model.state_dict(), 'best_model.pth')

✅ You're ready to train your model! 🚀
"""

print(next_steps)